# 03 - Text Chunking: The Most Underrated Step

Chunking is arguably the **most impactful** step in a RAG pipeline. The chunk size and strategy directly determine:

- **Retrieval precision**: Can we find the exact passage that answers the question?
- **Context completeness**: Does the retrieved chunk contain enough information to be useful?
- **Cost**: Larger chunks mean more tokens sent to the LLM.

This notebook compares four chunking strategies and visualizes the impact of different chunk sizes.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import matplotlib.pyplot as plt
import pandas as pd

from rag_engine.chunking.strategies import chunk_documents
from rag_engine.loaders import load_documents

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

## Load a Sample Document

We'll use our RAG survey markdown file — it's long enough to demonstrate meaningful differences.

In [ ]:
docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
full_text = docs[0].page_content
print(f"Document length: {len(full_text)} characters")
print(f"Word count: ~{len(full_text.split())} words")

## Strategy 1: Character Text Splitter (Naive)

Splits at a fixed character count. Simple but can cut mid-sentence or mid-word.

In [ ]:
character_chunks = chunk_documents(docs, strategy="character", chunk_size=512, chunk_overlap=50)

print(f"Number of chunks: {len(character_chunks)}")
print("\n--- Chunk 1 ---")
print(character_chunks[0].page_content)
print("\n--- Chunk 2 ---")
print(character_chunks[1].page_content[:200] + "...")

## Strategy 2: Recursive Character Splitter (Recommended Default)

Tries to split on paragraph boundaries (`\n\n`), then sentences (`\n`), then spaces. This preserves semantic coherence much better than naive splitting.

In [ ]:
recursive_chunks = chunk_documents(docs, strategy="recursive", chunk_size=512, chunk_overlap=50)

print(f"Number of chunks: {len(recursive_chunks)}")
print("\n--- Chunk 1 ---")
print(recursive_chunks[0].page_content)
print("\n--- Chunk 2 ---")
print(recursive_chunks[1].page_content[:200] + "...")

## Strategy 3: Token-Based Splitter

Splits based on token count rather than character count. This is useful because LLMs have **token** limits, not character limits. A 512-token chunk maps directly to LLM context usage.

In [ ]:
token_chunks = chunk_documents(docs, strategy="token", chunk_size=128, chunk_overlap=20)

print(f"Number of chunks: {len(token_chunks)}")
print("\n--- Chunk 1 ---")
print(token_chunks[0].page_content)

## Comparing Chunk Size Distributions

Let's visualize how each strategy distributes chunk sizes.

In [ ]:
strategies = {
    "Character": character_chunks,
    "Recursive": recursive_chunks,
    "Token": token_chunks,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, chunks) in zip(axes, strategies.items()):
    lengths = [len(c.page_content) for c in chunks]
    ax.hist(lengths, bins=15, edgecolor="black", alpha=0.7, color="steelblue")
    ax.set_title(f"{name} Splitter")
    ax.set_xlabel("Chunk length (chars)")
    ax.set_ylabel("Count")
    ax.axvline(x=sum(lengths)/len(lengths), color="red", linestyle="--", label=f"Mean: {sum(lengths)//len(lengths)}")
    ax.legend()

plt.suptitle("Chunk Size Distribution by Strategy", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Experiment: Varying Chunk Size

The choice of `chunk_size` has a huge impact. Let's see how it affects the number of chunks and their average length.

In [ ]:
chunk_sizes = [128, 256, 512, 1024, 2048]
results = []

for size in chunk_sizes:
    chunks = chunk_documents(docs, strategy="recursive", chunk_size=size, chunk_overlap=size // 10)
    lengths = [len(c.page_content) for c in chunks]
    results.append({
        "chunk_size": size,
        "num_chunks": len(chunks),
        "avg_length": sum(lengths) // len(lengths),
        "min_length": min(lengths),
        "max_length": max(lengths),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(df["chunk_size"], df["num_chunks"], "o-", color="steelblue", linewidth=2)
ax1.set_xlabel("Chunk Size (chars)")
ax1.set_ylabel("Number of Chunks")
ax1.set_title("Chunk Count vs. Chunk Size")

ax2.plot(df["chunk_size"], df["avg_length"], "o-", color="coral", linewidth=2)
ax2.set_xlabel("Chunk Size (chars)")
ax2.set_ylabel("Average Chunk Length (chars)")
ax2.set_title("Avg Chunk Length vs. Chunk Size")

plt.tight_layout()
plt.show()

## Guidelines for Choosing Chunk Size

| Chunk Size | Best For | Trade-off |
|-----------|---------|----------|
| **128-256** | Precise Q&A, FAQ-style | May miss context needed for complex answers |
| **512** | General-purpose (recommended starting point) | Good balance of precision and context |
| **1024-2048** | Long-form analysis, summarization | Less precise retrieval, higher LLM cost |

**Chunk overlap** (typically 10-20% of chunk size) ensures that information at chunk boundaries isn't lost.

**Key insight:** There is no universally best chunk size — it depends on your data and use case. Always experiment and evaluate (see notebook 08).

**Next:** [04 - Embeddings and Vector Stores](./04_embeddings_and_vectorstores.ipynb)